In [2]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from io import StringIO
import os
import time
from IPython.display import clear_output

In [2]:
HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
VALID_SEASONS = ['2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']
BASE_URL = 'https://www.transfermarkt.de'


### Get all teams

In [ ]:
# Get all teams participating in DFB Pokal since 2014-15 season


for season in VALID_SEASONS:
    teams = []
    teams_links = []
    link = BASE_URL + f'dfb-pokal/startseite/pokalwettbewerb/DFB?saison_id={season}'
    response = requests.get(link, headers=HEADERS)
    soup = BeautifulSoup(response.content, 'html.parser')


    # Find all 'tr' tags with the class 'begegnungZeile'
    rows = soup.find_all('tr', class_='begegnungZeile')

    for row in rows:
        home_team = row.find('td', class_='verein-heim')
        away_team = row.find('td', class_='verein-gast')
        if home_team and away_team:
            home_team_title = home_team.find('a')['title']
            home_team_link = home_team.find('a')['href']
            away_team_title = away_team.find_all('a')[1]['title']
            away_team_link = away_team.find_all('a')[1]['href']
            teams_links.append(home_team_link[:-4])
            teams_links.append(away_team_link[:-4])
            teams.append(home_team_title)
            teams.append(away_team_title)

    if len(teams_links) == 0:
        print(f"No teams found for {season} season")
        continue
    else:
        print(f"Found {len(set(teams_links))} teams in {season} season")


    df = pd.DataFrame({'team': teams, 'link': teams_links, 'season': season})
    # Remove duplicates
    df = df.drop_duplicates(subset=['team', 'link'])
    # Reset index
    df = df.reset_index(drop=True)
    # Save to CSV
    df.to_csv(f'datasets/teams/{season}.csv', index=False)



Found 64 teams in 2014 season
Found 64 teams in 2015 season
Found 64 teams in 2016 season
Found 64 teams in 2017 season
Found 64 teams in 2018 season
Found 64 teams in 2019 season
Found 64 teams in 2020 season
Found 64 teams in 2021 season
Found 64 teams in 2022 season
Found 64 teams in 2023 season
Found 64 teams in 2024 season


In [13]:
# Path to the folder containing the CSV files
folder_path = 'datasets/teams/'

# List to store dataframes
dataframes = []

# Iterate through all files in the folder
for file_name in os.listdir(folder_path):
    if file_name.endswith('.csv'):
        file_path = os.path.join(folder_path, file_name)
        # Read the CSV file and append it to the list
        dataframes.append(pd.read_csv(file_path))

# Combine all dataframes into a single dataframe
combined_df = pd.concat(dataframes, ignore_index=False)

# remove duplicates
combined_df = combined_df.drop_duplicates(subset=['team'])
# Reset index
combined_df = combined_df.reset_index(drop=True)

# Display the combined dataframe
teams_df = combined_df.drop(columns=['season'])

teams_df.to_csv('datasets/teams/teams.csv', index=False)

In [5]:
# Get additional data for each team

teams_df = pd.read_csv('datasets/teams/teams.csv')
new_data = pd.DataFrame(columns=['link', 'stadium_name', 'total_capacity', 'lawn_heating', 'field_length', 'field_width'])

for index, row in teams_df.iterrows():
    print(f"{index}",end=' ')
    additional_data = {
        'link': '',
        'stadium_name': '',
        'total_capacity': '',
        'lawn_heating': '',
        'field_length': '',
        'field_width': ''
    }

    team_link = row['link']
    final_link = BASE_URL + team_link.replace('spielplan','stadion') + '2014'
    print(final_link)
    response = requests.get(final_link, headers=HEADERS)
    soup = BeautifulSoup(response.content, 'html.parser')
    table = soup.find('table', class_='profilheader')

    # Extract the data from the table
    additional_data['link'] = team_link
    additional_data['stadium_name'] = table.find('th', string='Stadionname:').find_next('td').text.strip()
    try:
        additional_data['total_capacity'] = table.find('th', string='Gesamtkapazität:').find_next('td').text.strip()
    except AttributeError:
        additional_data['total_capacity'] = ''
    try:
        additional_data['lawn_heating'] = table.find('th', string='Rasenheizung:').find_next('td').text.strip()
    except AttributeError:
        additional_data['lawn_heating'] = ''
    
    try:
        playing_surface = table.find('th', string='Spielfeldgröße:').find_next('td').text.strip()
        additional_data['field_length'] = playing_surface.split('x')[0].strip().split('m')[0]
        additional_data['field_width'] = playing_surface.split('x')[1].strip().split('m')[0]
    except AttributeError:
        additional_data['field_length'] = ''
        additional_data['field_width'] = ''

    # Append the data to the new dataframe
    new_data = pd.concat([new_data, pd.DataFrame([additional_data])], ignore_index=True)



0 https://www.transfermarkt.de/borussia-dortmund/stadion/verein/16/saison_id/2014
1 https://www.transfermarkt.de/vfl-wolfsburg/stadion/verein/82/saison_id/2014
2 https://www.transfermarkt.de/fc-bayern-munchen/stadion/verein/27/saison_id/2014
3 https://www.transfermarkt.de/arminia-bielefeld/stadion/verein/10/saison_id/2014
4 https://www.transfermarkt.de/sc-freiburg/stadion/verein/60/saison_id/2014
5 https://www.transfermarkt.de/tsg-1899-hoffenheim/stadion/verein/533/saison_id/2014
6 https://www.transfermarkt.de/borussia-monchengladbach/stadion/verein/18/saison_id/2014
7 https://www.transfermarkt.de/bayer-04-leverkusen/stadion/verein/15/saison_id/2014
8 https://www.transfermarkt.de/vfr-aalen/stadion/verein/83/saison_id/2014
9 https://www.transfermarkt.de/1-fc-kaiserslautern/stadion/verein/2/saison_id/2014
10 https://www.transfermarkt.de/sg-dynamo-dresden/stadion/verein/129/saison_id/2014
11 https://www.transfermarkt.de/1-fc-koln/stadion/verein/3/saison_id/2014
12 https://www.transfermark

In [7]:
# Merge the new data with teams_df on the 'link' column
updated_teams_df = teams_df.merge(new_data, on='link', how='left')

# Display the updated dataframe
updated_teams_df

# Save the updated dataframe to a CSV file
updated_teams_df.to_csv('datasets/teams/teams.csv', index=False)

### Get all players

In [ ]:
positions = {
    'Torwart': 'GK',
    'Innenverteidiger': 'CB',
    'Abwehr': 'CB',
    'Mittelfeld': 'Midfielder',
    'Linker Verteidiger' : 'LB',
    'Rechter Verteidiger' : 'RB',
    'Defensives Mittelfeld' : 'CDM',
    'Zentrales Mittelfeld' : 'CM',
    'Linkes Mittelfeld' : 'LM',
    'Rechtes Mittelfeld' : 'RM',
    'Offensives Mittelfeld' : 'CAM',
    'Rechtsaußen': 'RW',
    'Linksaußen': 'LW',
    'Mittelstürmer': 'ST',
    'Sturm': 'ST',
    'Hängende Spitze': 'Second Striker',
    'Libero': 'Sweeper',
}

# List of all scraped teams
scraped_teams = [os.path.splitext(f)[0] for f in os.listdir('datasets/players') if f.endswith('.csv')]
scraped_teams.append('SV Drochtersen/Assel')
scraped_teams.append('1.FC Germania Egestorf/Langreder')

teams_df = pd.read_csv('datasets/teams/teams.csv')

for index, row in teams_df.iterrows():
    data = []
    selected_team = row
    if selected_team['team'] in scraped_teams:
        continue
    print(f"Scraping {selected_team['team']}:")
    for season in VALID_SEASONS:
        print(season)
        if season != '2024':
            final_link = BASE_URL + selected_team['link'].replace('spielplan','kader').replace('saison_id/','plus/0/galerie/0?saison_id=') + season
        else: 
            final_link = BASE_URL + selected_team['link'].replace('spielplan','startseite').replace('saison_id/','')
        while True:
                response = requests.get(final_link, headers=HEADERS)
                if response.content and b'Service Unavailable' not in response.content and response.status_code == 200:
                    break
                print("Empty or error response, retrying in 5 seconds...")
                time.sleep(5)
            
        soup = BeautifulSoup(response.content, 'html.parser')
        table = soup.find('table', class_='items')

        # Extract the data from the table
        try:
            rows = table.find_all('tr')[1::3]  # Skip the header row
        except:
            print(f"No data found for {selected_team['team']} in {season} season")
            continue
        for row in rows:
            cols = row.find_all('td')
            if len(cols) > 0:
                try:
                    if season != '2024':
                        player_data = {
                            'name': cols[1].text.strip().split("\n")[0].strip(),
                            'position': positions[cols[1].text.strip().split("\n")[-1].strip()],
                            'age': cols[5].text.strip(),
                            'value': cols[8].text.strip(),
                            'team': selected_team['team'],
                            'season': season,
                        }
                    else:
                        player_data = {
                            'name': cols[1].text.strip().split("\n")[0].strip(),
                            'position': positions[cols[1].text.strip().split("\n")[-1].strip()],
                            'age': cols[5].text.split(' ')[1][1:-1],
                            'value': cols[7].text.strip(),
                            'team': selected_team['team'],
                            'season': season,
                        }
                    data.append(player_data)
                except KeyError:
                    continue

    df = pd.DataFrame(data)
    # Remove duplicates
    df = df.drop_duplicates(subset=['name', 'season'])
    # Reset index
    df = df.reset_index(drop=True)
    # Save to CSV
    if '/' in selected_team['team']:
        df.to_csv(f'datasets/players/{selected_team['team'].replace('/','-')}.csv', index=False)
    else:
        df.to_csv(f'datasets/players/{selected_team['team']}.csv', index=False)

    print(f"Saved {selected_team['team']} data to CSV")
    print(f"Sleeping for 1 minutes to avoid overloading the server...")
    time.sleep(60)
    clear_output(wait=True)

In [ ]:
# Path to the folder containing the CSV files
folder_path = 'datasets/players/'

# List to store dataframes
dataframes = []

# Iterate through all files in the folder
for file_name in os.listdir(folder_path):
    if file_name.endswith('.csv'):
        file_path = os.path.join(folder_path, file_name)
        # Read the CSV file and append it to the list
        dataframes.append(pd.read_csv(file_path))

# Combine all dataframes into a single dataframe
combined_df = pd.concat(dataframes, ignore_index=False)

# Reset index
combined_df = combined_df.reset_index(drop=True)

combined_df.to_csv('datasets/players/players.csv', index=False)

combined_df

### Team Schedules: TODO

In [21]:
# team schedule
teams_df = pd.read_csv('datasets/teams/teams.csv')
scraped_teams = [os.path.splitext(f)[0] for f in os.listdir('datasets/schedules') if f.endswith('.csv')]
for index, row in teams_df.iterrows():
    selected_team = row
    if selected_team['team'] in scraped_teams:
        continue
    print(f"Scraping {selected_team['team']}:")

    data = []
    for season in VALID_SEASONS:
        final_link = BASE_URL + selected_team['link'] + season
        print(season, final_link)


        # Fetch the webpage using requests
        response = requests.get(final_link, headers=HEADERS)

        # Parse the page source with BeautifulSoup
        soup = BeautifulSoup(response.text, 'html.parser')
        if season == '2024':
            competitions = soup.find_all('div', class_='box')[3:-2]
        else:
            competitions = soup.find_all('div', class_='box')[2:-2]
        for competition in competitions:
            # Find the title of the competition
            title = competition.find('a').text.strip()
            table = competition.find('table')
            if table:
                rows = table.find_all('tr')[1:]  # Skip the header row
                for row in rows:
                    cols = row.find_all('td')
                    if len(cols) > 0:
                        try:
                            date = cols[1].text.split('. ')[1].strip()
                            time = cols[2].text.strip()
                        except:
                            date = ''
                            time = ''
                        host = cols[3].text.strip()
                        if host == 'H':
                            home_team = selected_team['team']
                            away_team = cols[6].text.strip().split('(')[0].strip()
                            home_rank = cols[4].text.strip()[1:].split('.')[0]
                            try:
                                away_rank = cols[6].text.strip().split('(')[1].split('.')[0].strip()
                            except:
                                away_rank = ''
                        else:
                            home_team = cols[6].text.strip().split('(')[0].strip()
                            away_team = selected_team['team']
                            try:
                                home_rank = cols[6].text.strip().split('(')[1].split('.')[0].strip()
                            except:
                                home_rank = ''
                            away_rank = cols[4].text.strip()[1:].split('.')[0]
                        
                        participants = cols[8].text.strip()
                        result = cols[9].text.strip()
                        try:
                            home_score = result.split(':')[0].strip()
                            away_score = result.split(':')[1].split(' ')[0].strip()
                        except:
                            home_score = ''
                            away_score = ''

                        match_link = cols[9].find('a')['href']
                        match_data = {
                            'date': date,
                            'time': time,
                            'home_team': home_team,
                            'away_team': away_team,
                            'participants': participants,
                            'home_score': home_score,
                            'away_score': away_score,
                            'match_link': match_link,
                            'season': season,
                            'competition': title,
                        }
                        # Append the data to the list
                        data.append(match_data)
    df = pd.DataFrame(data)
    clear_output(wait=True)

    df.to_csv(f'datasets/schedules/{selected_team['team'].replace('/','-')}.csv', index=False)

Scraping VfV Borussia 06 Hildesheim:
2014 https://www.transfermarkt.de/vfv-borussia-06-hildesheim/spielplan/verein/2540/saison_id/2014
2015 https://www.transfermarkt.de/vfv-borussia-06-hildesheim/spielplan/verein/2540/saison_id/2015
2016 https://www.transfermarkt.de/vfv-borussia-06-hildesheim/spielplan/verein/2540/saison_id/2016
2017 https://www.transfermarkt.de/vfv-borussia-06-hildesheim/spielplan/verein/2540/saison_id/2017
2018 https://www.transfermarkt.de/vfv-borussia-06-hildesheim/spielplan/verein/2540/saison_id/2018
2019 https://www.transfermarkt.de/vfv-borussia-06-hildesheim/spielplan/verein/2540/saison_id/2019
2020 https://www.transfermarkt.de/vfv-borussia-06-hildesheim/spielplan/verein/2540/saison_id/2020
True
2021 https://www.transfermarkt.de/vfv-borussia-06-hildesheim/spielplan/verein/2540/saison_id/2021
2022 https://www.transfermarkt.de/vfv-borussia-06-hildesheim/spielplan/verein/2540/saison_id/2022
2023 https://www.transfermarkt.de/vfv-borussia-06-hildesheim/spielplan/verei

In [18]:
# Path to the folder containing the CSV files
folder_path = 'datasets/schedules/'

# List to store dataframes
dataframes = []

# Iterate through all files in the folder
for file_name in os.listdir(folder_path):
    if file_name.endswith('.csv'):
        file_path = os.path.join(folder_path, file_name)
        # Read the CSV file and append it to the list
        dataframes.append(pd.read_csv(file_path))

# Combine all dataframes into a single dataframe
combined_df = pd.concat(dataframes, ignore_index=False)


# Remove duplicates
combined_df = combined_df.drop_duplicates()

# Reset index
combined_df = combined_df.reset_index(drop=True)


combined_df.to_csv('datasets/schedules/schedules.csv', index=False)

combined_df

,date,time,home_team,away_team,participants,home_score,away_score,match_link,season,competition
0,27.08.2014,19:00,Dyn. Dresden II,1. FC Lokomotive Leipzig,732,0,1,/spielbericht/index/spielbericht/2477944,2014,NOFV-Oberliga Süd
1,08.08.2014,19:30,1. FC Lokomotive Leipzig,Markranstädt,3.214,0,0,/spielbericht/index/spielbericht/2479773,2014,NOFV-Oberliga Süd
2,23.08.2014,14:00,FCO Neugersdorf,1. FC Lokomotive Leipzig,719,0,0,/spielbericht/index/spielbericht/2479780,2014,NOFV-Oberliga Süd
3,31.08.2014,14:00,1. FC Lokomotive Leipzig,Schott Jena,2.224,2,1,/spielbericht/index/spielbericht/2479791,2014,NOFV-Oberliga Süd
4,12.09.2014,18:30,1. FC Lokomotive Leipzig,Askan. Bernburg,2.237,1,0,/spielbericht/index/spielbericht/2479802,2014,NOFV-Oberliga Süd
...,...,...,...,...,...,...,...,...,...,...
56848,17.05.2025,14:00,FV Illertissen,Würzburger Kickers,NaN,0.0,1.0,/spielbericht/index/spielbericht/4387020,2024,Regionalliga Bayern
56849,16.08.2024,18:00,Würzburger Kickers,TSG Hoffenheim,9.511,5.0,7.0,/spielbericht/index/spielbericht/4353899,2024,DFB-Pokal
56850,06.08.2024,18:30,Sittenbachtal,Würzburger Kickers,800,1.0,9.0,/spielbericht/index/spielbericht/4408926,2024,Landespokal Bayern
56851,20.08.2024,18:30,Würzburger Kickers,Aschaffenburg,970,6.0,5.0,/spielbericht/index/spielbericht/4430549,2024,Landespokal Bayern


In [ ]:
start_lineup = []

link = 'https://www.transfermarkt.de/1-fc-kaiserslautern_fc-st-pauli/aufstellung/spielbericht/4096275'
response = requests.get(link, headers=HEADERS)
soup = BeautifulSoup(response.content, 'html.parser')


lineups = soup.find_all('table', class_='items')

for lineup in lineups:
    rows = lineup.find_all('tr')[1::3]

    for row in rows:
      player_names = row.find('a', class_='wichtig')
      print(player_names['title'])
      